# Step-sampling scheme: align the training distribution with the TS-AUC weight

## The mismatch

TS-AUC weights each online step by `n_pos * n_neg` at that step. Early steps are almost
all unbroken (`n_pos ~ 0`, weight ~ 0); late steps are almost all broken (`n_neg ~ 0`).
The weight peaks **mid-series**, where positives and negatives coexist.

The deployed trainer samples ~40 **log-spaced** steps per series - dense early, sparse late.
Measured on the cache, that puts **73% of training rows in the earliest 20% of the timeline,
which carries only 13% of the metric weight**, and starves the mid-series steps that carry
73%. Correlation(step metric-weight, step train-rows) = -0.011. The model is trained hardest
where the metric barely looks.

This is not the "denser training" dead-end (that added rows everywhere and lost). Same row
**budget**, relocated toward where the metric scores.

## Variants (all label-free, so the mask is global with zero leakage)

Each samples ~40 steps/series, matching the deployed budget - only the **spacing** changes:
- **S0_log** current log-spacing (baseline)
- **S1_uniform** evenly spaced in step index
- **S2_triangular** density peaks at mid-series (proxy for the `n_pos*n_neg` curve)
- **S3_late** density rises toward late steps

Pointwise CatBoost (the reverted base after YetiRank was dropped), same 5 folds as
`cv_revalidate`, `paired_compare` vs S0. A gain is only believed at **t >= 2**.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

d = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = d['X'], d['y'], d['sid'], d['time']
log_mask = d['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
folds = CV.series_folds(sid, n_splits=5, seed=0)

# series boundaries (cache is (id,time)-sorted, so each series is a contiguous block)
uids, starts = np.unique(sid, return_index=True)
bounds = np.append(starts, len(sid))
lengths = np.diff(bounds)
print(f'{X.shape[0]:,} rows | {len(uids):,} series | log-mask rows {log_mask.sum():,}')

5,036,517 rows | 10,000 series | log-mask rows 328,996


In [2]:
def sample_steps(L, m, kind):
    """Return up to m distinct step indices in [0, L) drawn with the given spacing.

    Deterministic (uses inverse-CDF on a fixed grid, not RNG) so masks are reproducible.
    All variants return ~m indices, so training budget is held roughly constant."""
    if L <= m:
        return np.arange(L)
    if kind == 'log':
        return np.unique(np.round(np.expm1(np.linspace(0, np.log1p(L - 1), m))).astype(int))
    if kind == 'uniform':
        return np.unique(np.round(np.linspace(0, L - 1, m)).astype(int))
    # density-based: build a CDF over fractions u in [0,1], invert on m evenly spaced probs
    u = np.linspace(0, 1, 512)
    if kind == 'triangular':
        dens = 1.0 - np.abs(2 * u - 1)           # peak at u=0.5
    elif kind == 'late':
        dens = u                                 # rises toward u=1
    else:
        raise ValueError(kind)
    cdf = np.cumsum(dens); cdf /= cdf[-1]
    probs = (np.arange(m) + 0.5) / m
    frac = np.interp(probs, cdf, u)
    return np.unique(np.round(frac * (L - 1)).astype(int))


def build_mask(kind, m=40):
    """Boolean mask over cache rows marking the sampled training steps for `kind`."""
    mask = np.zeros(len(sid), bool)
    for k in range(len(uids)):
        s, L = bounds[k], lengths[k]
        mask[s + sample_steps(L, m, kind)] = True
    return mask


MASKS = {'S0_log': log_mask,
         'S1_uniform': build_mask('uniform'),
         'S2_triangular': build_mask('triangular'),
         'S3_late': build_mask('late')}

# sanity: my rebuilt 'log' should match the cache's stored mask bit-for-bit
assert np.array_equal(build_mask('log'), log_mask), 'log rebuild != cached mask'
for n, m in MASKS.items():
    pr = y[m].mean()
    print(f'{n:16s} rows {m.sum():>8,} | sampled pos-rate {pr:.3f}')

S0_log           rows  328,996 | sampled pos-rate 0.111
S1_uniform       rows  394,922 | sampled pos-rate 0.254
S2_triangular    rows  393,586 | sampled pos-rate 0.255
S3_late          rows  393,759 | sampled pos-rate 0.334


In [3]:
CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)

def cat_fp(train, val):
    m = CatBoostClassifier(**CATP)
    m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

RESULTS_JSON = 'sampling_results.json'
# resumable: reload finished configs, only run the rest (harness fix - a 6h ranker run
# earlier gave zero interim output because nbconvert buffers stdout to the end).
done = json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}
res = {}
for name, mask in MASKS.items():
    t = time.time()
    print(f'\n=== {name} ===', flush=True)
    r = CV.run_cv(X, y, sid, index, cat_fp, folds=folds, train_row_mask=mask, row_step=step)
    res[name] = r
    done[name] = dict(fold_scores=r.fold_scores.tolist(), mean=r.mean, std=r.std,
                      sem=r.sem, oof=r.oof_score)
    json.dump(done, open(RESULTS_JSON, 'w'), indent=2)   # write AFTER each config
    print(f'{r.summary(name)} | {time.time() - t:.0f}s', flush=True)


=== S0_log ===


  fold 0: train   263,351 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5849


  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5964


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5828


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6014


...

  fold 1: train   315,078 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5960


  fold 2: train   314,845 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5931


  fold 3: train   314,994 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6074


  fold 4: train   314,840 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6226


S3_late            mean 0.6028 +/- 0.0124 (sem 0.0055) | OOF 0.6023 | folds: 0.5950  0.5960  0.5931  0.6074  0.6226 | 308s


In [4]:
print(CV.summarize(res))
print('\n--- each scheme vs S0_log baseline ---')
for n in res:
    if n != 'S0_log':
        print()
        print(CV.paired_compare(res[n], res['S0_log'], n, 'S0_log'))

S3_late            mean 0.6028 +/- 0.0124 (sem 0.0055) | OOF 0.6023 | folds: 0.5950  0.5960  0.5931  0.6074  0.6226
S1_uniform         mean 0.5988 +/- 0.0151 (sem 0.0068) | OOF 0.5981 | folds: 0.5946  0.5968  0.5795  0.6016  0.6215
S0_log             mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232
S2_triangular      mean 0.5948 +/- 0.0154 (sem 0.0069) | OOF 0.5941 | folds: 0.5862  0.5908  0.5770  0.6037  0.6163

--- each scheme vs S0_log baseline ---

S1_uniform - S0_log: +0.0011 +/- 0.0051 (sem 0.0023, t +0.48, wins 3/5) -> within noise
  per-fold deltas: +0.0097  +0.0004  -0.0033  +0.0002  -0.0017

S2_triangular - S0_log: -0.0029 +/- 0.0044 (sem 0.0020, t -1.49, wins 2/5) -> within noise
  per-fold deltas: +0.0013  -0.0055  -0.0058  +0.0024  -0.0069

S3_late - S0_log: +0.0051 +/- 0.0054 (sem 0.0024, t +2.12, wins 3/5) -> likely
  per-fold deltas: +0.0101  -0.0004  +0.0103  +0.0060  -0.0006
